# Notebook 3: Generate Reports

**VLM-ARB Cloud Framework**

This notebook fetches results from Firestore and generates comprehensive reports with visualizations.

## Workflow
1. Setup: Authenticate with Firebase
2. Fetch evaluation results from Firestore
3. Generate visualizations (bar charts, heatmaps)
4. Create PDF reports
5. Upload reports to Cloud Storage
6. Generate shareable links for team

**Key Feature**: Shareable public links - all team members can view reports without additional auth.

---

## Cell 1: Install Dependencies & Setup

In [ ]:
# Install dependencies
import subprocess
import sys

packages = [
    'firebase-admin',
    'matplotlib',
    'seaborn',
    'numpy',
    'pandas',
    'reportlab',
    'pillow',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("✅ All dependencies installed")

## Cell 2: Setup Authentication & Fetch Results

In [ ]:
## Cell 2: Setup & Fetch Results

from pathlib import Path
from datetime import datetime, timedelta
import json
import firebase_admin
from firebase_admin import credentials, firestore

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

team_folder = Path("/content/drive/MyDrive/VLM-ARB-Team")
print("📊 Environment Setup:")
print(f"  Team Folder: {team_folder}")
print(f"  Google Drive: ✅ Mounted")

# Initialize Firebase (optional)
fs = None
firebase_available = False
creds_path = team_folder / "secrets" / "serviceAccountKey.json"

if creds_path.exists():
    try:
        if not firebase_admin._apps:
            creds = credentials.Certificate(str(creds_path))
            firebase_admin.initialize_app(creds)
        fs = firestore.client()
        firebase_available = True
        print("✅ Firebase authenticated")
    except Exception as e:
        print(f"⚠️  Firebase failed: {str(e)[:100]}")
        print("   Using Drive-only results")
else:
    print("⚠️  Firebase credentials not found - using Drive-only")

# Fetch results from Google Drive
print("\n📊 Fetching evaluation results...")

runs = []
results_folder = team_folder / "results" / "raw"

if results_folder.exists():
    try:
        # Find all JSON result files
        result_files = list(results_folder.glob("*.json"))
        print(f"   Found {len(result_files)} result files on Drive")
        
        for result_file in result_files:
            try:
                with open(result_file, 'r') as f:
                    run_data = json.load(f)

                    runs.append(run_data)print(f"\n📋 Available Runs: {len(runs)}")

            except:

                continue    print("\n⚠️  Using sample data for demo report")

            runs = [sample_run]

        if runs:    }

            print(f"✅ Loaded {len(runs)} evaluations from Drive")        }

    except Exception as e:            'mobilevit': {'synthetic_asr': 0.55, 'coco_asr': 0.45, 'robustness_gap': 0.10}

        print(f"⚠️  Error reading Drive results: {e}")            'clip': {'synthetic_asr': 0.45, 'coco_asr': 0.35, 'robustness_gap': 0.10},

else:        'synthetic_vs_coco': {

    print(f"   ℹ️  Results folder not found - will use sample data")        },

            'llava': {'asr': 0.78, 'ods': 0.65, 'sbr': 0.12},

# If no runs found, create sample data for demo            'blip2': {'asr': 0.68, 'ods': 0.58, 'sbr': 0.05},

if not runs:            'mobilevit': {'asr': 0.45, 'ods': 0.38, 'sbr': 0.0},

    sample_run = {            'clip': {'asr': 0.35, 'ods': 0.28, 'sbr': 0.0},

        'run_id': f'eval_sample_{datetime.now().strftime("%Y%m%d_%H%M%S")}',        'metrics': {
        'timestamp': datetime.now().isoformat(),

## Cell 3: Select Run(s) & Display Results

In [ ]:
import pandas as pd

# Auto-select latest run (or you can manually select)
if runs:
    selected_run = runs[0]  # Latest run
    selected_run_id = selected_run.get('run_id', 'unknown')
    
    print(f"\n📊 Selected Run: {selected_run_id}")
    
    # Extract metrics
    metrics = selected_run.get('metrics', {})
    print(f"\n📈 Results Summary:")
    
    # Create DataFrame for display
    results_data = []
    for model_name, model_metrics in metrics.items():
        if isinstance(model_metrics, dict) and 'asr' in model_metrics:
            results_data.append({
                'Model': model_name,
                'ASR': f"{model_metrics.get('asr', 0):.3f}",
                'ODS': f"{model_metrics.get('ods', 0):.3f}",
                'SBR': f"{model_metrics.get('sbr', 0):.3f}",
                'CMCS': f"{model_metrics.get('cmcs', 'N/A')}"
            })
    
    if results_data:
        df = pd.DataFrame(results_data)
        print(df.to_string(index=False))
    else:
        print("No model results in selected run")
else:
    print("No runs available")
    selected_run = None

## Cell 4: Generate Visualizations (Bar Charts)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

if selected_run:
    # Setup output directory
    figures_dir = Path("/tmp/vlm_arb_figures")
    figures_dir.mkdir(exist_ok=True)
    
    # Extract model names and metrics
    metrics = selected_run.get('metrics', {})
    models = []
    asr_values = []
    ods_values = []
    sbr_values = []
    
    for model_name, model_metrics in metrics.items():
        if isinstance(model_metrics, dict) and 'asr' in model_metrics:
            models.append(model_name.upper())
            asr_values.append(model_metrics.get('asr', 0))
            ods_values.append(model_metrics.get('ods', 0))
            sbr_values.append(model_metrics.get('sbr', 0))
    
    if models:
        print("📊 Generating visualization: Model Robustness Comparison")
        
        # Create 3-panel comparison chart
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(f'VLM-ARB Model Robustness Evaluation\n{selected_run_id}', 
                     fontsize=14, fontweight='bold', y=1.02)
        
        # ASR (Attack Success Rate)
        axes[0].bar(models, asr_values, color='#FF6B6B', edgecolor='black', linewidth=1.5)
        axes[0].set_title('Attack Success Rate (ASR)\n(Higher = More Vulnerable)', fontweight='bold')
        axes[0].set_ylabel('Score (0-1)')
        axes[0].set_ylim([0, 1.0])
        axes[0].grid(axis='y', alpha=0.3)
        for i, v in enumerate(asr_values):
            axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
        axes[0].tick_params(axis='x', rotation=45)
        
        # ODS (Output Deviation Score)
        axes[1].bar(models, ods_values, color='#4ECDC4', edgecolor='black', linewidth=1.5)
        axes[1].set_title('Output Deviation Score (ODS)\n(Higher = More Change)', fontweight='bold')
        axes[1].set_ylabel('Score (0-1)')
        axes[1].set_ylim([0, 1.0])
        axes[1].grid(axis='y', alpha=0.3)
        for i, v in enumerate(ods_values):
            axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
        axes[1].tick_params(axis='x', rotation=45)
        
        # SBR (Safety Bypass Rate)
        axes[2].bar(models, sbr_values, color='#95E1D3', edgecolor='black', linewidth=1.5)
        axes[2].set_title('Safety Bypass Rate (SBR)\n(Higher = More Bypassed)', fontweight='bold')
        axes[2].set_ylabel('Score (0-1)')
        axes[2].set_ylim([0, 1.0])
        axes[2].grid(axis='y', alpha=0.3)
        for i, v in enumerate(sbr_values):
            axes[2].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')
        axes[2].tick_params(axis='x', rotation=45)

            print("⏭️  Skipping visualization (no selected run)")

        plt.tight_layout()else:

                print("⚠️  No model metrics found in results")

        # Save figure    else:

        chart_path = figures_dir / "01_model_comparison.png"            plt.close()

        plt.savefig(chart_path, dpi=150, bbox_inches='tight')            print(f"✅ Chart saved: {gap_chart_path.name}")

        print(f"✅ Chart saved: {chart_path.name}")            plt.savefig(gap_chart_path, dpi=150, bbox_inches='tight')

        plt.close()            gap_chart_path = figures_dir / "02_robustness_gap.png"

                    

        # Generate Synthetic vs COCO comparison if available            plt.tight_layout()

        if 'synthetic_vs_coco' in selected_run:            

            print("\n📊 Generating visualization: Synthetic vs Real Images")                ax.text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

                        for i, v in enumerate(gap_values):

            synth_vs_coco = selected_run['synthetic_vs_coco']            

            gap_models = []            ax.legend()

            gap_values = []            ax.axhline(y=0.1, color='red', linestyle='--', alpha=0.5, label='Significant gap threshold')

                        ax.grid(axis='y', alpha=0.3)

            for model_name, comparison in synth_vs_coco.items():            ax.set_ylabel('Robustness Gap (ASR difference)', fontweight='bold')

                gap_models.append(model_name.upper())                        fontweight='bold', fontsize=12)

                gap_values.append(comparison.get('robustness_gap', 0))            ax.set_title('Robustness Gap: Synthetic vs COCO Images\n(Positive = Synthetic More Vulnerable)', 

                        ax.bar(gap_models, gap_values, color=colors, edgecolor='black', linewidth=1.5)

            fig, ax = plt.subplots(figsize=(10, 5))            colors = ['#FF6B6B' if gap > 0.1 else '#FFD93D' for gap in gap_values]

## Cell 5: Create PDF Report

In [ ]:
import shutil
import json

if selected_run:
    print("\n💾 Saving all reports to Google Drive...\n")
    
    # Create reports directory on Google Drive
    drive_reports_dir = Path("/content/drive/MyDrive/VLM-ARB-Team/reports")
    drive_reports_dir.mkdir(parents=True, exist_ok=True)
    
    # Copy all chart images to Drive
    saved_count = 0
    for chart_file in figures_dir.glob("*.png"):
        drive_chart_path = drive_reports_dir / f"{selected_run_id}_{chart_file.name}"
        shutil.copy(str(chart_file), str(drive_chart_path))
        print(f"✅ Chart: {chart_file.name}")
        saved_count += 1
    
    # Save evaluation results as JSON
    metrics_data = {
        'run_id': selected_run_id,
        'timestamp': datetime.now().isoformat(),
        'models': list(selected_run.get('metrics', {}).keys()),
        'metrics': selected_run.get('metrics', {}),
        'synthetic_vs_coco': selected_run.get('synthetic_vs_coco', {}),
        'notes': 'Full evaluation results with synthetic and real-world (COCO) image comparisons'
    }
    
    json_path = Path("/tmp/eval_results.json")
    with open(json_path, 'w') as f:
        json.dump(metrics_data, f, indent=2, default=str)
    
    drive_json_path = drive_reports_dir / f"{selected_run_id}_results.json"
    shutil.copy(str(json_path), str(drive_json_path))
    print(f"✅ JSON: {selected_run_id}_results.json")
    
    # Create a markdown summary report
    markdown_content = f"""# VLM-ARB Evaluation Report

    print("⏭️  No data to save")

**Run ID**: {selected_run_id}else:

**Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}    print(f"   Results: 1 (JSON)")

    print(f"   Reports: 1 (Markdown)")

## Executive Summary    print(f"   Charts: {saved_count}")

    print(f"\n📊 Summary:")

This report presents the evaluation results of Vision Language Models (VLMs) against image injection attacks.    print(f"   📁 {drive_reports_dir}")

    print(f"\n✅ All reports saved to Google Drive:")

## Model Robustness Results    

    print(f"✅ JSON: {selected_run_id}_results.json")

| Model | ASR | ODS | SBR |    print(f"✅ Report: {selected_run_id}_report.md")

|-------|-----|-----|-----|    shutil.copy(str(md_path), str(drive_md_path))

"""    drive_md_path = drive_reports_dir / f"{selected_run_id}_report.md"

        

    for model_name, model_metrics in selected_run.get('metrics', {}).items():        f.write(markdown_content)

        if isinstance(model_metrics, dict) and 'asr' in model_metrics:    with open(md_path, 'w') as f:

            asr = model_metrics.get('asr', 0)    md_path = Path("/tmp/report.md")

            ods = model_metrics.get('ods', 0)    

            sbr = model_metrics.get('sbr', 0)"""

            markdown_content += f"| {model_name.upper()} | {asr:.3f} | {ods:.3f} | {sbr:.3f} |\n"*Report generated by VLM-ARB Cloud Framework*

    

    markdown_content += f"""---



## Attack Metrics Explanation- Larger models (BLIP-2, LLaVA) show higher vulnerability

- Real-world (COCO) images provide better robustness assessment

- **ASR (Attack Success Rate)**: Percentage of images where the attack changed the model's output- Models are more vulnerable to attacks on synthetic images

- **ODS (Output Deviation Score)**: Measure of how much the output changed (0-1)

- **SBR (Safety Bypass Rate)**: Percentage of attacks that bypassed safety guidelines## Key Findings



## Visualizations- **Attack Types**: Prompt injection (HOUYI) + Typographic overlays

- **Real Images**: COCO 2017 dataset subset + attack variants

### Model Comparison- **Synthetic Images**: 5 base images + attack variants

- `01_model_comparison.png`: Side-by-side comparison of ASR, ODS, and SBR across models

- `02_robustness_gap.png`: Difference in robustness between synthetic and real (COCO) images## Dataset Information


## Cell 6: Upload Reports to Cloud Storage

In [ ]:
print("\n" + "="*60)
print("✅ REPORT GENERATION COMPLETE")
print("="*60)

drive_reports_dir = Path("/content/drive/MyDrive/VLM-ARB-Team/reports")

if drive_reports_dir.exists():
    print(f"\n📂 Reports saved to Google Drive:")
    print(f"   📁 {drive_reports_dir}")
    
    print(f"\n📋 Report Files:")
    for report_file in sorted(drive_reports_dir.glob(f"{selected_run_id}*")):
        print(f"   📄 {report_file.name}")
    
    print(f"\n🔗 To share reports with team:")
    print(f"   1. Open Google Drive")
    print(f"   2. Navigate to VLM-ARB-Team/reports")
    print(f"   3. Right-click on PDF file → Share")
    print(f"   4. Set access to 'Anyone with link' or 'Your team'")
    print(f"   5. Copy link and share via Slack/email")
    
    print(f"\n💡 Tip: Share the entire 'reports' folder with your team")
    print(f"   for easy access to all historical reports")
else:
    print("⚠️  Reports directory not found")

print("="*60)

## Cell 7: Generate Shareable Links & Display Results

In [ ]:
print("\n" + "="*60)
print("✅ REPORT GENERATION COMPLETE")
print("="*60)

if selected_run:
    print(f"\n📊 Report Details:")
    print(f"   Run ID: {selected_run_id}")
    print(f"   Timestamp: {datetime.now().isoformat()}")
    print(f"   Models: {len(models)}")
    
    if pdf_url:
        print(f"\n🔗 Shareable Links (paste in team chat/email):")
        print(f"   PDF Report: {pdf_url}")
        if chart_url:
            print(f"   Chart: {chart_url}")
        if json_url:
            print(f"   Results (JSON): {json_url}")
    else:
        print(f"\n📂 Local Files:")
        print(f"   PDF: {pdf_path}")
        print(f"   Chart: {figures_dir}/model_comparison.png")
    
    print(f"\n📊 Metrics Summary:")
    for model_name in models:
        model_metrics = metrics.get(model_name, {})
        print(f"   {model_name}: ASR={model_metrics.get('asr', 0):.3f}, ODS={model_metrics.get('ods', 0):.3f}, SBR={model_metrics.get('sbr', 0):.3f}")
else:
    print("No results to display")

print("="*60)

## Cell 8: Optional - Browse All Historical Reports

In [ ]:
print("\n📚 All Generated Reports:")
print("\nTo view all reports, fetch from Cloud Storage:")

if fs and not fs.offline_mode:
    all_reports = fs.list_files("reports/")
    print(f"\nFound {len(all_reports)} files in Cloud Storage:")
    for report_file in all_reports[:10]:  # Show first 10
        file_name = Path(report_file).name
        url = fs.get_public_url(report_file)
        print(f"   • {file_name}")
else:
    print("Not connected to Cloud Storage")